# NeuroRAG Ablation Study on MMLU

This notebook allows you to disable (ablate) almost any component of the NeuroRAG pipeline and evaluate the effect on MMLU accuracy.

In [ ]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import json
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  summac_zs_metric,
  summac_conv_metric,
  bert_score_metric,
)

/Users/vladimirskvortsov/Projects/neurorag/notebooks/metrics.py:40: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model='llama3.1')
2025-08-10 16:47:13,490 - INFO - Using default tokenizer.
2025-08-10 16:47:13,490 - INFO - Using default tokenizer.


## Disable warnings

In [49]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

In [50]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'OPENAI_PROXY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Ablation Config
Set any component to False to disable it in the pipeline.

In [51]:
ablation_config: dict[str, bool] = {
    'step_back': True,
    'query_rewriting': True,
    'decomposition': True,
    'hyde': True,
    'vector_store': True,
    'pubmed': True,
    'arxiv': True,
    'ncbi_protein': True,
    'ncbi_gene': True,
    'biorxiv': True,
    'medrxiv': True,
    'document_grading': True,
    'hallucination_grading': True,
    'answer_grading': True,
    'web_search': True,
}


## NeuroRAG Wrapper for Ablation
This class disables components according to the config above.

In [52]:
class NeuroRAGAblation(NeuroRAG):
    def __init__(self, ablation_config, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ablation_config = ablation_config

    def generate_step_back_query_node(self, state):
        if not self.ablation_config.get('step_back', True):
            return {'step_back_query': state['query']}
        return super().generate_step_back_query_node(state)

    def generate_rewritten_query_node(self, state):
        if not self.ablation_config.get('query_rewriting', True):
            return {'rewritten_query': state['query']}
        return super().generate_rewritten_query_node(state)

    def generate_subqueries_node(self, state):
        if not self.ablation_config.get('decomposition', True):
            return {'subqueries': []}
        return super().generate_subqueries_node(state)

    def generate_hyde_documents_node(self, state):
        if not self.ablation_config.get('hyde', True):
            return {'generated_documents': [state['query']]}
        return super().generate_hyde_documents_node(state)

    def vector_store_retriever_node(self, state):
        if not self.ablation_config.get('vector_store', True):
            return {'documents': []}
        return super().vector_store_retriever_node(state)

    def pub_med_retriever_node(self, state):
        if not self.ablation_config.get('pubmed', True):
            return {'documents': []}
        return super().pub_med_retriever_node(state)

    def arxiv_retriever_node(self, state):
        if not self.ablation_config.get('arxiv', True):
            return {'documents': []}
        return super().arxiv_retriever_node(state)

    def ncbi_protein_db_retriever_node(self, state):
        if not self.ablation_config.get('ncbi_protein', True):
            return {'documents': []}
        return super().ncbi_protein_db_retriever_node(state)

    def ncbi_gene_db_retriever_node(self, state):
        if not self.ablation_config.get('ncbi_gene', True):
            return {'documents': []}
        return super().ncbi_gene_db_retriever_node(state)

    def biorxiv_retriever_node(self, state):
        if not self.ablation_config.get('biorxiv', True):
            return {'documents': []}
        return super().biorxiv_retriever_node(state)

    def medrxiv_retriever_node(self, state):
        if not self.ablation_config.get('medrxiv', True):
            return {'documents': []}
        return super().medrxiv_retriever_node(state)

    def grade_documents_node(self, state):
        if not self.ablation_config.get('document_grading', True):
            # Bypass grading, just pass all documents through
            return {'documents': state['documents'], 'web_search': False}
        return super().grade_documents_node(state)

    def grade_generation_node(self, state):
        if not self.ablation_config.get('hallucination_grading', True) and not self.ablation_config.get('answer_grading', True):
            return 'useful'
        if not self.ablation_config.get('hallucination_grading', True):
            # Only answer grading
            query = state['query']
            generation = state['generation']
            try:
                grade = self.answer_grade_chain.invoke(query, generation)
            except Exception:
                grade = 'no'
            return 'useful' if grade == 'yes' else 'not useful'
        if not self.ablation_config.get('answer_grading', True):
            # Only hallucination grading
            documents = state['documents']
            generation = state['generation']
            try:
                context = (
                    '\n\n' + '\n\n'.join(map(lambda doc: doc.page_content, documents)) + '\n\n'
                )
                grade = self.hallucinations_chain.invoke(generation, context)
            except Exception:
                grade = 'no'
            return 'useful' if grade == 'yes' else 'not useful'
        return super().grade_generation_node(state)

    def web_search_node(self, state):
        if not self.ablation_config.get('web_search', True):
            return {'documents': [], 'web_results': []}
        return super().web_search_node(state)


## Setup evaluation

### Load cache

In [ ]:
try:
  with open('cache.json', 'r') as file:
    cache = json.load(file)
except FileNotFoundError:
  cache = {}

### Load QA dataset

In [ ]:
mediqa_df = pd.read_csv('../datasets/neurobiology_mediqa.csv')
mediqa_df

,question,answer
0,SSPE. My son is 33years of age and did not hav...,Subacute sclerosing panencephalitis: Subacute ...
1,Homozygout MTHFR A1298C Health Issues and long...,MTHFR gene variant (Inheritance): Because each...
2,What is Stroke?,Stroke: A stroke occurs when the blood supply ...
3,What causes Stroke?,Ischemic Stroke (Summary): Summary A stroke is...
4,What are the symptoms of Stroke?,What are the symptoms of Stroke?: The signs an...
5,What are the treatments of Stroke?,Stroke (Treatment): A stroke is a medical emer...
6,What is Dementia?,Dementia (WHAT IS DEMENTIA?): Dementia is the ...
7,What causes Dementia?,What causes Dementia?: Dementia usually occurs...
8,What are the symptoms of Dementia?,Dementia (Symptoms): Dementia symptoms include...
9,How to diagnose Dementia?,Dementia (Diagnosis): Diagnosing dementia and ...


### Evaluation function

In [ ]:
import time

def invoke_with_retries(app, question, k = 10, sleep: int = 5):
    for i in range(k):
        try:
            return app.invoke(question)
        except Exception as e:
            if i == k - 1:
                raise e
            else:
                print(e)
            time.sleep(sleep * i)

In [ ]:
def eval_rag(app, experiment_name) -> float:
    questions = list(mediqa_df['question'].tolist())
    expected_answers = list(mediqa_df['answer'].tolist())
    predicted_answers = []

    if 'ablation_study' not in cache:
      cache['ablation_study'] = {}

    if experiment_name not in cache['ablation_study']:
      cache['ablation_study'][experiment_name] = {}

    for index, question in tqdm(enumerate(questions)):
        if question not in cache['ablation_study'][experiment_name]:
          response = invoke_with_retries(app, question)
          cache['ablation_study'][experiment_name][question] = response['generation']

        predicted_answers.append(cache['ablation_study'][experiment_name][question])

    cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
    bleu_score = bleu_metric(expected_answers, predicted_answers)
    rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
    rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
    factscore_score = factscore_metric(expected_answers, predicted_answers)
    summac_zs_score = summac_zs_metric(expected_answers, predicted_answers)
    summac_conv_score = summac_conv_metric(expected_answers, predicted_answers)
    bert_score_score = bert_score_metric(expected_answers, predicted_answers)

    return cos_score, bleu_score, rogue_1_score, rogue_l_score, factscore_score, summac_zs_score, summac_conv_score, bert_score_score

## Run Ablation Study
Try a few ablation settings and compare results.

In [ ]:
ablation_settings = [
    ('All enabled', ablation_config.copy()),
    ('No step_back', {**ablation_config, 'step_back': False}),
    ('No query_rewriting', {**ablation_config, 'query_rewriting': False}),
    ('No decomposition', {**ablation_config, 'decomposition': False}),
    ('No hyde', {**ablation_config, 'hyde': False}),
    ('No document_grading', {**ablation_config, 'document_grading': False}),
    ('No hallucination_grading', {**ablation_config, 'hallucination_grading': False}),
    ('No answer_grading', {**ablation_config, 'answer_grading': False}),
    ('No web_search', {**ablation_config, 'web_search': False}),
]

results = []
for name, config in ablation_settings:
    print(f'Running: {name}')
    app = NeuroRAGAblation(config, debug=False)
    app.compile()

    (
        cos_score,
        bleu_score,
        rogue_1_score,
        rogue_l_score,
        factscore_score,
        summac_zs_score,
        summac_conv_score,
        bert_score_score,
    ) = eval_rag(app, name)

    results.append({
        'experiment_name': name,
        'cos_score': cos_score,
        'bleu_score': bleu_score,
        'rogue_1_score': rogue_1_score,
        'rogue_l_score': rogue_l_score,
        'factscore_score': factscore_score,
        'summac_zs_score': summac_zs_score,
        'summac_conv_score': summac_conv_score,
        'bert_score_score': bert_score_score,
    })
    print(f'{name}: cos_score = {cos_score}, bleu_score = {bleu_score}, rogue_1_score = {rogue_1_score}, rogue_l_score = {rogue_l_score}, factscore_score = {factscore_score}, summac_zs_score = {summac_zs_score}, summac_conv_score = {summac_conv_score}, bert_score_score = {bert_score_score}')

pd.DataFrame(results)